# Model Training
I trained an interpretable logistic regression model to predict enterprise non-renewal risk three months before contract end using leakage-safe features derived from recent engagement, support burden, satisfaction, and quarter-over-quarter deterioration. I evaluated the baseline using ROC-AUC and PR-AUC and inspected feature coefficients to understand the strongest drivers of churn risk.

In [0]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from pyspark.ml.functions import vector_to_array

In [0]:
renewal_model_dataset = spark.table("renewal_model_dataset")

In [0]:
print("renewal_model_dataset rows:", renewal_model_dataset.count())

In [0]:
# Inspect label balance: important because churn problems are often imbalanced 
display(
    renewal_model_dataset
    .groupBy("non_renewal_flag")
    .count()
)

**I checked label balance because churn problems are often imbalanced, and that affects both evaluation and decision thresholds. In my case, non-renewals were about 20% of the sample, which is meaningfully imbalanced but not extreme. So I kept a simple baseline model, but evaluated it with PR-AUC and churn-focused metrics rather than relying on accuracy alone.**

Imbalance affects how you evaluate the model, which baseline you compare against, whether you rebalance traning and how you choose thresholds

**How much imbalance is relevant?**

How I think about it is 
- Around 40/60 or 30/70 is mild imbalance 
- Around 20/80 or 10/90 is meaningful imbalance 
- Below 5/95 is severe imbalance 
- Below 1/99 is extreme imbalance

**Practical Question to Ask**

*Do I have enough positive examples to learn patterns?*

In this example, I have 51 non-renewals, it is small but enough for a portfolio case. I should be careful about overfitting and avoid models too complex too early.

When imbalance is severe or extreme **accuracy** becomes misleading. Let's say my model predicts that everyone will renew, in this example, the model would be right 80% of the time. This could lead me to believe that the model is good. **The problem?** That this model is no caching any churners. 

**What do you do once you detect imbalnce?** 
1. *You change how you evaluate the model*

You should care less about plain **accuracy** and more about: 
- PR-AUC
- Precision
- Recall 
- Recall at top K 
- Sometimes F1 

Because the business question is "Can I find the non-renewals?" not "Can I classify the majority class well?"

2. *You compare against the right baseline* 

For PR-AUC, the naive baseline is roughly the positive class prevalence. In this example, it would be ~0.20. You would check if PR-AUC is meaningfully bette than that baseline.

3. *You may change the training approach*

When imbalance gets stronger, you may consider: class weighting, oversampling the minority class, undersampling the majority class, threshold tuning. In this example, I don't consider I have to rebalance yet but I should be aware of the imbalance. 

4. *You change how you choose the decision threshold*

Default classification threshold is often 0.5. For churn, you might prefer a lower threshold if the business wants to catch more at-risk acocunts. 
- **0.5**: fewer flagged accounts, higher precision (minimizes false positives, in this case it would minimize the cases where it predicts churn but the client didn't churn)
- **0.3**: more flagged accounts, higher recall (minimizes false negatives, in this case it would minimize the cases where it predicts renewal but the client churned)

In [0]:
display(
    renewal_model_dataset.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in renewal_model_dataset.columns]
    )
)

In [0]:
# No nulls present but will still "fill" numeric missing values with simple defaults
model_df = renewal_model_dataset.fillna({
    "utilization_avg_prior_3m": 0.0,
    "sessions_avg_prior_3m": 0.0,
    "active_members_avg_prior_3m": 0.0,
    "webinar_attendance_avg_prior_3m": 0.0,
    "employer_comms_avg_prior_3m": 0.0,
    "ticket_count_avg_prior_3m": 0.0,
    "high_severity_tickets_avg_prior_3m": 0.0,
    "nps_avg_prior_3m": 0.0,
    "utilization_change_3m_vs_prior_3m": 0.0,
    "sessions_change_3m_vs_prior_3m": 0.0,
    "active_members_change_3m_vs_prior_3m": 0.0,
    "webinar_attendance_change_3m_vs_prior_3m": 0.0,
    "employer_comms_change_3m_vs_prior_3m": 0.0,
    "ticket_count_change_3m_vs_prior_3m": 0.0,
    "high_severity_tickets_change_3m_vs_prior_3m": 0.0,
    "nps_change_3m_vs_prior_3m": 0.0,
    "active_member_rate_last_3m": 0.0,
    "sessions_per_active_member_last_3m": 0.0,
    "tickets_per_1000_eligible_last_3m": 0.0
})

In [0]:
# Choose features: I use a mix of recent-state features, change features, ratios and categorical context 
numeric_features = [
    "eligible_employees",
    "annual_contract_value",
    "contract_term_months",
    
    "utilization_avg_last_3m",
    "sessions_avg_last_3m",
    "active_members_avg_last_3m",
    "webinar_attendance_avg_last_3m",
    "employer_comms_avg_last_3m",
    "ticket_count_avg_last_3m",
    "high_severity_tickets_avg_last_3m",
    "nps_avg_last_3m",

    "utilization_avg_prior_3m",
    "sessions_avg_prior_3m",
    "active_members_avg_prior_3m",
    "webinar_attendance_avg_prior_3m",
    "employer_comms_avg_prior_3m",
    "ticket_count_avg_prior_3m",
    "high_severity_tickets_avg_prior_3m",
    "nps_avg_prior_3m",

    "utilization_change_3m_vs_prior_3m",
    "sessions_change_3m_vs_prior_3m",
    "active_members_change_3m_vs_prior_3m",
    "webinar_attendance_change_3m_vs_prior_3m",
    "employer_comms_change_3m_vs_prior_3m",
    "ticket_count_change_3m_vs_prior_3m",
    "high_severity_tickets_change_3m_vs_prior_3m",
    "nps_change_3m_vs_prior_3m",

    "active_member_rate_last_3m",
    "sessions_per_active_member_last_3m",
    "tickets_per_1000_eligible_last_3m"
]

categorical_features = [
    "industry",
    "region",
    "company_size"
]

I used **StringIndexer** to convert categorical variables into numeric indices as required by Spark ML. I set **handleInvalid**="keep" to ensure robustness to unseen categories during scoring. The indexed values were then **one-hot encoded** to avoid imposing artificial ordinal relationships.

In [0]:
# Build features transformers to convert categorical variables into numeric vectors 
indexers = [
    StringIndexer(inputCol=c, outputCol = f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]

#StringIndexer creates a list of transformers that convert categorical text columns into numeric indices (integers). By default, Spark assigns indices on category frequency: most frequent category -> index 0. Example: Healthcare = 0, Tech = 1, Retail = 2. 
# Because I used list comprehension "for c in categorical_features" then three indexers get created (one for industry, one for region and one for company size), each one outputs a new column.
# Spark ML pipelines (pipeline is nothing else than an ordered sequence of transformations applied to your data: raw data -> preprocessing -> feature vector -> model -> predictions) expect a sequence of stages, so each indexer becomes one stage in the pipeline
# handleInvalid = "keep" tells Spark what to do with missing values and unseen categories at the prediction time (without it the model would crash when seeing a new category, like Education)

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]
#Indexing alone is not enough because integer encoding implies ordering (Retail > Tech > Healthcare), which could lead to the odel learning nonsense relationships. That is why I used OneHotEncoder whic converts integer to binary vector, for example: Healthcare [1,0,0], Tech [0,1,0], Retail [0,0,1].
# OneHotEncoder expects numeric category indeces, not strings. It can't operate on text. Therefore, the need of using StringIndexer first


In [0]:
# Assemble final feature vector 
assembler_inputs = numeric_features + [f"{c}_ohe" for c in categorical_features]

assemble = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features", 
    handleInvalid="keep"
)
# VectorAssemble combines all numeric features into one vector column that includes numeric variables, one-hot vectors, rations, trends, etc. Models in Spark expect exactly one column of type Vector

In [0]:
# Define the model 
lr = LogisticRegression(
    featuresCol="features",
    labelCol="non_renewal_flag",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=100,
)
# I used the default of 100 iterations to ensure convergence while avoiding unnecessary computation. For small datasets like this one, far fewer iterations would typically suffice, but using the default is a safe baseline.

In [0]:
# Build the pipeline 
pipeline = Pipeline(stages=indexers + encoders + [assemble, lr])

In [0]:
# Train/test split
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed = 42)

print("train rows:", train_df.count())
print("test rows:", test_df.count())

In [0]:
# fit the model 
model = pipeline.fit(train_df)

In [0]:
#Score the test set 
predictions = model.transform(test_df)

display(predictions
        .select(
            "client_id"
            ,"renewal_cycle_id"
            ,"non_renewal_flag"
            ,"prediction"
            ,"probability"
        )
)

In [0]:
# Evaluate model performance 
roc_evaluator = BinaryClassificationEvaluator(
    labelCol = "non_renewal_flag", 
    rawPredictionCol = "rawPrediction", 
    metricName = "areaUnderROC"
)

roc_auc = roc_evaluator.evaluate(predictions)
print(f"ROC AUC: {roc_auc}")

In [0]:
#PR-AUC
pr_evaluator = BinaryClassificationEvaluator(
    labelCol = "non_renewal_flag", 
    rawPredictionCol = "rawPrediction", 
    metricName = "areaUnderPR"
)

pr_auc = pr_evaluator.evaluate(predictions)
print("PR AUC:", pr_auc)

The baseline model demonstrates strong discriminatory power (ROC-AUC ≈ 0.90) and materially improves identification of non-renewals over random ranking (PR-AUC ≈ 0.35), indicating that early engagement deterioration signals are predictive of renewal risk three months in advance.

In [0]:
# Create a readable risk score
predictions_scored = predictions.withColumn(
    "non_renewal_probability",
    vector_to_array("probability")[1]
)

display(
    predictions_scored.select(
    "client_id"
    ,"renewal_cycle_id"
    ,"non_renewal_flag"
    ,"prediction"
    ,"non_renewal_probability"
    )
    .orderBy(F.desc("non_renewal_probability"))
    
)

In [0]:
# Extract model coefficients 
lr_model = model.stages[-1]

# The fitted pipeline returns a PipelineModel containing all trained stages, including preprocessing transformers and the final estimator. The logistic regression model is the last stage, so I extract it using model.stages[-1] to access coefficients and intercept.

print("Intercept:", lr_model.intercept)
print("Number of Coefficients:", len(lr_model.coefficients))

# The intercept represents the baseline log-odds of non-renewal when all predictors are at their reference values (no deterioration). Converting −2.09 to probability yields roughly 11%, indicating that a typical client without risk signals is unlikely to churn, which aligns with enterprise renewal dynamics. Intercept negative means less than 50% of probability

In [0]:
# import math 
# log_odds = lr_model.intercept
# prob= 1 / (1 + math.exp(-log_odds))
# print(prob)

In [0]:
# Map coefficients back to feature names 

feature_names = model.stages[-2].getInputCols()
coefficients = lr_model.coefficients.toArray()

coef_df = spark.createDataFrame(
    list(zip(feature_names, [float(x) for x in coefficients])),
    schema=["feature", "coefficient"]
)

# I used zip() to pair each feature name with its corresponding coefficient because the model returns coefficients as a vector without metadata. This allows building a readable table of feature importance. I used list() because zip() returns an iterator (not a full list)

# Inspect strongest positive coeffients 
display(coef_df.orderBy(F.desc("coefficient")).limit(10))

In [0]:
# Inspect strongest negative coeffients 
display(coef_df.orderBy(F.asc("coefficient")).limit(10))

- **Positive coefficient:** increases non-renewal risk
- **Negative coefficient:** descreases non-renewal risk

In [0]:
spark.sql("DROP TABLE IF EXISTS renewal_model_predictions")
predictions_scored.write.mode("overwrite").saveAsTable("renewal_model_predictions")
spark.sql("DROP TABLE IF EXISTS renewal_model_coefficients")
coef_df.write.mode("overwrite").saveAsTable("renewal_model_coefficients")